# Step 1 - Load and explore the dataset

In [2]:
!pip install kagglehub

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("surekharamireddy/fake-news-detection")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fake-news-detection' dataset.
Path to dataset files: /kaggle/input/fake-news-detection


In [7]:
import os
os.listdir(path)

['train_news.csv']

In [22]:
from data_loader import load_dataset, show_distribution
df = load_dataset(path + "/train_news.csv")

df.head()

,Unnamed: 0,id,headline,written_by,news,label
0,0,9653,Ethics Questions Dogged Agriculture Nominee as...,Eric Lipton and Steve Eder,"WASHINGTON — In Sonny Perdue’s telling, Geo...",0
1,1,10041,U.S. Must Dig Deep to Stop Argentina’s Lionel ...,David Waldstein,HOUSTON — Venezuela had a plan. It was a ta...,0
2,2,19113,Cotton to House: ’Do Not Walk the Plank and Vo...,Pam Key,"Sunday on ABC’s “This Week,” while discussing ...",0
3,3,6868,"Paul LePage, Besieged Maine Governor, Sends Co...",Jess Bidgood,"AUGUSTA, Me. — The beleaguered Republican g...",0
4,4,7596,A Digital 9/11 If Trump Wins,Finian Cunningham,Finian Cunningham has written extensively on...,1


In [23]:
show_distribution(df)

label
1    10413
0    10387
Name: count, dtype: int64


# Step 2 - Split the dataset

In [24]:
# vectorizing before splitting lets the vectorizer learn vocabulary from the entire dataset, including the test data.
# This means the model indirectly sees information from the test set - Data Leakage. That males the evaluation unfair.
# so split dataset is step 2.


In [29]:
from sklearn.model_selection import train_test_split

df = df.dropna(subset=["news"])

X = df["news"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 3 - Vectorize

In [30]:
from preprocess import vectorize_text

X_train_vec, X_test_vec, vectorizer = vectorize_text(X_train, X_test)

# Step 4 - Build model

In [32]:
from model import build_model

model = build_model(X_train_vec.shape[1])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │       640,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 648,449 (2.47 MB)

 Trainable params: 648,449 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

# Step 5 - Train Model

In [33]:
from train import train_model

history = train_model(model, X_train_vec, y_train)

Epoch 1/15
468/468 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9124 - loss: 0.2102 - val_accuracy: 0.9530 - val_loss: 0.1328
Epoch 2/15
468/468 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9804 - loss: 0.0611 - val_accuracy: 0.9476 - val_loss: 0.1609
Epoch 3/15
468/468 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9920 - loss: 0.0258 - val_accuracy: 0.9494 - val_loss: 0.1807
Epoch 4/15
468/468 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9970 - loss: 0.0100 - val_accuracy: 0.9446 - val_loss: 0.2217
Epoch 5/15
468/468 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9977 - loss: 0.0072 - val_accuracy: 0.9446 - val_loss: 0.2595
Epoch 6/15
468/468 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9985 - loss: 0.0051 - val_accuracy: 0.9476 - val_loss: 0.2614
Epoch 7/15
468/468 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9986 - loss: 0.0035 - val_accuracy: 0.9446 - val_loss: 0.2885
Epoch 8/15
468/468 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9979 - loss: 0.0061 - val_accu

# Step 6 - Evaluate

In [34]:
loss, accuracy = model.evaluate(X_test_vec, y_test)

print("Accuracy:", accuracy)

130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9420 - loss: 0.3820
Accuracy: 0.941969633102417


# Step 7 - Test custom headlines

In [35]:
from predict import predict_news

headline = "Scientists confirm water on Mars"

predict_news(model, vectorizer, headline)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step


'Fake News'